# NFL Exploration Project

In [1]:
import nflreadpy as nfl
import pandas as pd

In [2]:
stats_2025 = nfl.load_player_stats([2025])
print(stats_2025.head())
stats_2025.write_csv("../data/player_stats_2025.csv")

shape: (5, 150)
┌───────────┬───────────┬───────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ player_id ┆ player_na ┆ player_di ┆ position ┆ … ┆ pt_return ┆ pt_net_ya ┆ fantasy_p ┆ fantasy_p │
│ ---       ┆ me        ┆ splay_nam ┆ ---      ┆   ┆ _tds      ┆ rds       ┆ oints     ┆ oints_ppr │
│ str       ┆ ---       ┆ e         ┆ str      ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│           ┆ str       ┆ ---       ┆          ┆   ┆ i32       ┆ i32       ┆ f64       ┆ f64       │
│           ┆           ┆ str       ┆          ┆   ┆           ┆           ┆           ┆           │
╞═══════════╪═══════════╪═══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 00-002345 ┆ A.Rodgers ┆ Aaron     ┆ QB       ┆ … ┆ 0         ┆ 0         ┆ 25.66     ┆ 25.66     │
│ 9         ┆           ┆ Rodgers   ┆          ┆   ┆           ┆           ┆           ┆           │
│ 00-002385 ┆ M.Prater  ┆ Matt      ┆ K        ┆ … ┆ 0         ┆ 0         

## Defining the replacement values

In [4]:
# Use regular season only
stats_2025 = nfl.load_player_stats([2025]).to_pandas()
stats = stats_2025[stats_2025["season_type"] == "REG"].copy()

# Keep fantasy-relevant positions
positions = ["QB", "RB", "WR", "TE"]
stats = stats[stats["position"].isin(positions)]

# Aggregate weekly rows into season totals
season_totals = (
    stats
    .groupby(["player_id", "player_display_name", "position"], as_index=False)
    .agg(
        games=("week", "nunique"),
        fantasy_points_ppr=("fantasy_points_ppr", "sum"),
        fantasy_points=("fantasy_points", "sum"),
    )
)

replacement_levels = {
    "QB": 10, # one starter per team, 10 teams
    "RB": 25, # two rbs per team plus some flex
    "WR": 30, # 2-3 WR slots + flex demand
    "TE": 10, # one starter per team
}

replacement_points = {}

for position, rank in replacement_levels.items():
    position_players = (
        season_totals[season_totals["position"] == position]
        .sort_values("fantasy_points_ppr", ascending=False)
        .reset_index(drop=True)
    )

    replacement_points[position] = position_players.loc[
        rank - 1, "fantasy_points_ppr"
    ]

season_totals["replacement_points"] = season_totals["position"].map(replacement_points)

season_totals["vorp"] = (
    season_totals["fantasy_points_ppr"] - season_totals["replacement_points"]
)

vorp_rankings = season_totals.sort_values("vorp", ascending=False)

vorp_rankings.head(25)

,player_id,player_display_name,position,games,fantasy_points_ppr,fantasy_points,replacement_points,vorp
48,00-0033280,Christian McCaffrey,RB,17,416.60,314.60,178.80,237.80
426,00-0039075,Puka Nacua,WR,16,375.00,246.00,180.90,194.10
357,00-0038542,Bijan Robinson,RB,17,370.80,291.80,178.80,192.00
427,00-0039139,Jahmyr Gibbs,RB,17,366.90,289.90,178.80,188.10
179,00-0036223,Jonathan Taylor,RB,17,362.30,316.30,178.80,183.50
358,00-0038543,Jaxon Smith-Njigba,WR,17,359.90,240.90,180.90,179.00
417,00-0039040,De'Von Achane,RB,16,322.80,255.80,178.80,144.00
247,00-0036963,Amon-Ra St. Brown,WR,17,324.00,207.00,180.90,143.10
314,00-0037744,Trey McBride,TE,17,315.90,189.90,177.70,138.20
239,00-0036900,Ja'Marr Chase,WR,16,313.60,188.60,180.90,132.70
